# 03_audio.ipynb — UC-3 음성 다운로드

`audio()` 함수와 재사용, 포맷 선택, 예외 검증.


In [ ]:
import os
import youtube_kit as yk

from pathlib import Path
# 루트의 outputs 에 저장한다. notebooks 에서 열어도, 루트에서 실행해도 같은 곳이다
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
out_dir = str(ROOT / "outputs")
os.makedirs(out_dir, exist_ok=True)


## 포맷 확인 및 다운로드 (T-13 Shorts)


In [2]:
# info() 결과에서 오디오 포맷 선택 (하드코딩하지 않음)
info_t13 = yk.info("JyEsEcsCqzM", out_dir=out_dir)
a_id = next(f["id"] for f in info_t13["audio_formats"] if f["ext"] == "m4a")
print("Selected audio format:", a_id)

res_audio = yk.audio("JyEsEcsCqzM", audio=a_id, out_dir=out_dir)
print("audio() result:", res_audio)
assert res_audio["status"] in ("ok", "reused")
assert res_audio["path"] == os.path.join(out_dir, f"JyEsEcsCqzM.{a_id}.m4a")
assert os.path.exists(res_audio["path"])


Selected audio format: 139-drc
audio() result: {'status': 'ok', 'video_id': 'JyEsEcsCqzM', 'url': 'https://www.youtube.com/watch?v=JyEsEcsCqzM', 'path': 'outputs\\JyEsEcsCqzM.139-drc.m4a'}


## 파일 재사용 및 overwrite 확인


In [3]:
res_re = yk.audio("JyEsEcsCqzM", audio=a_id, out_dir=out_dir)
assert res_re["status"] == "reused"

res_ow = yk.audio("JyEsEcsCqzM", audio=a_id, out_dir=out_dir, overwrite=True)
assert res_ow["status"] == "ok"
assert os.path.exists(res_ow["path"])
print("Reuse and overwrite verified.")


Reuse and overwrite verified.


## T-1 (bWPXADZylm0): 원본 음성과 더빙 음성 포맷 선택 확인


In [4]:
info_t1 = yk.info("bWPXADZylm0", out_dir=out_dir)
orig_fmt = next(f for f in info_t1["audio_formats"] if f["original"] is True)
dub_fmt = next(f for f in info_t1["audio_formats"] if f["original"] is False)
print("T-1 original format:", orig_fmt["id"], orig_fmt["language"])
print("T-1 dubbed format:", dub_fmt["id"], dub_fmt["language"])
assert orig_fmt["id"] != dub_fmt["id"]


T-1 original format: 139-1 ko
T-1 dubbed format: 139-0 en-US


## T-7 (lbzSN3fFV_Q): DRC 변형 포맷 선택 확인


In [5]:
info_t7 = yk.info("lbzSN3fFV_Q", out_dir=out_dir)
drc_fmt = next(f for f in info_t7["audio_formats"] if f["drc"] is True)
print("T-7 DRC audio format:", drc_fmt["id"])
assert drc_fmt["drc"] is True


T-7 DRC audio format: 139-drc


## T-15: 라이브 영상 다운로드 시 NotAvailable


In [6]:
try:
    yk.audio("rFZHOHl-L8A", audio="140", out_dir=out_dir)
    assert False, "Expected NotAvailable"
except yk.NotAvailable as exc:
    print("T-15 live stream rejected in audio():", type(exc).__name__)


T-15 live stream rejected in audio(): NotAvailable


## 없는 포맷 ID (NotAvailable)


In [7]:
try:
    yk.audio("JyEsEcsCqzM", audio="99999", out_dir=out_dir)
    assert False, "Expected NotAvailable"
except yk.NotAvailable as exc:
    print("Non-existent audio format rejected:", type(exc).__name__)


Non-existent audio format rejected: NotAvailable


## 잘못된 인자 (InvalidInput)


In [8]:
try:
    yk.audio("JyEsEcsCqzM", audio="", out_dir=out_dir)
    assert False, "Expected InvalidInput"
except yk.InvalidInput as exc:
    print("Empty audio ID rejected:", type(exc).__name__)

try:
    yk.audio("https://www.youtube.com/playlist?list=PLyPc3E0Xm540Yfdln9qlVyyrqRME80jJm", audio="140", out_dir=out_dir)
    assert False, "Expected InvalidInput"
except yk.InvalidInput as exc:
    print("Playlist URL rejected in audio():", type(exc).__name__)


Empty audio ID rejected: InvalidInput
Playlist URL rejected in audio(): InvalidInput


## T-3 · T-4 · T-5 · T-7 · T-9 · T-12 음성 받기

더빙만, 셋 이상 중 하나, 한국어 더빙, DRC 변형, webm(변환 없음), 언어 없는 음성.

In [ ]:
cases = [
    ("VbD8ITrJ6lg", lambda f: f["language"] == "en-US" and f["original"] is False and f["ext"] == "m4a"),  # T-3
    ("DOTROnklS28", lambda f: f["language"] == "id" and f["ext"] == "m4a"),                                # T-4
    ("XQk8jNM4zD8", lambda f: f["language"] == "ko" and f["original"] is False and f["ext"] == "m4a"),     # T-5
    ("lbzSN3fFV_Q", lambda f: f["drc"] is True),                                                           # T-7
    ("SbvAaDN1bpE", lambda f: f["ext"] == "webm"),                                                          # T-9
    ("DDubtRhOEGw", lambda f: f["language"] is None and f["ext"] == "m4a"),                                # T-12
]
for vid, pick in cases:
    meta = yk.info(vid, out_dir=out_dir)
    f = next(f for f in meta["audio_formats"] if pick(f))
    r = yk.audio(vid, audio=f["id"], out_dir=out_dir)
    assert r["path"] == os.path.join(out_dir, f"{vid}.{f['id']}.{f['ext']}") and os.path.exists(r["path"])
    print(vid, f["id"], f["language"], f["ext"], r["status"])

## 진행률 콜백

`progress`에 함수를 넘기면 `download` 이벤트가 여러 번 오고, 파일을 최종 이름으로 옮긴 뒤 `done`이 온다.

In [ ]:
meta_t5 = yk.info("XQk8jNM4zD8", out_dir=out_dir)
a_t5 = next(f for f in meta_t5["audio_formats"] if f["ext"] == "m4a" and f["original"] is True and not f["drc"])

events = []
r = yk.audio("XQk8jNM4zD8", audio=a_t5["id"], out_dir=out_dir, overwrite=True, progress=events.append)
stages = [e["stage"] for e in events]
assert stages.count("download") >= 1 and stages[-1] == "done", stages
last_dl = [e for e in events if e["stage"] == "download"][-1]
print(r["status"], len(events), "events; last download:", last_dl["downloaded_bytes"], "/", last_dl["total_bytes"])

## 취소

`cancel`에 `threading.Event`를 넘기고 다운로드 도중에 set하면 `Cancelled`가 나고, 최종 파일도 임시 폴더도 남지 않는다. 첫 `download` 이벤트에서 set해 다운로드 한가운데서 멈추게 한다.

In [ ]:
import threading

target = os.path.join(out_dir, f"XQk8jNM4zD8.{a_t5['id']}.{a_t5['ext']}")
if os.path.exists(target):
    os.remove(target)
before = set(os.listdir(out_dir))

stop = threading.Event()
def stop_on_first_chunk(event):
    if event["stage"] == "download":
        stop.set()

try:
    yk.audio("XQk8jNM4zD8", audio=a_t5["id"], out_dir=out_dir, progress=stop_on_first_chunk, cancel=stop)
    assert False, "Expected Cancelled"
except yk.Cancelled as exc:
    print("Cancelled:", exc)

assert not os.path.exists(target)
assert set(os.listdir(out_dir)) == before, set(os.listdir(out_dir)) - before
print("No file or temporary folder left.")